# 第19章 数据质量检查与清洗

建立数据质量检查流程，处理缺失、重复、异常和无效记录。


## 先解决一个小问题

拿一组小型业务数据练习“数据质量检查与清洗”：先看数据结构，再完成一次明确的计算或转换。建立数据质量检查流程，处理缺失、重复、异常和无效记录。


## 这章为什么先学

这是“Pandas”路线中第 19 章的操作重点。本章只解决“数据质量检查与清洗”，不重复前面章节已经完成的准备工作。


## 开始前确认

- 掌握 Python 基础语法、列表和字典
- 开始前先确认：生成质量概览


## 做完要留下什么

产出一个与“数据质量检查与清洗”直接对应的结果，并记录输入形状、字段或筛选口径。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 生成质量概览
- 处理缺失值
- 识别并删除重复
- 使用规则标记异常


## 核心概念

- 先统计问题规模，再决定删除、填充或保留。
- 缺失值处理取决于业务含义，不能统一填0。
- 异常值应先标记和调查，不能机械删除。


## 示例 1：质量概览

同时查看形状、缺失、重复和类型。


In [ ]:
import numpy as np
import pandas as pd


orders = pd.DataFrame({
    "order_id": ["A1", "A2", "A2", "A3", "A4"],
    "region": ["华东", "华南", "华南", None, "华北"],
    "amount": [320.0, 880.0, 880.0, np.nan, 9800.0],
})
print("形状:", orders.shape)
print("缺失:\n", orders.isna().sum())
print("重复行:", orders.duplicated().sum())
print(orders.dtypes)


## 示例 2：缺失与重复处理

订单ID重复时需要明确保留规则。


In [ ]:
clean = orders.drop_duplicates(subset="order_id", keep="first").copy()
clean["region"] = clean["region"].fillna("未知")
median_amount = clean["amount"].median()
clean["amount"] = clean["amount"].fillna(median_amount)
print(clean)


## 示例 3：IQR异常标记

标记异常并保留原值，便于后续调查。


In [ ]:
q1 = clean["amount"].quantile(0.25)
q3 = clean["amount"].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr
clean["is_outlier"] = clean["amount"] > upper
print("上界:", upper)
print(clean[clean["is_outlier"]])


## 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd


# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates = ["InvoiceDate"],
    dtype = {"InvoiceNo": "string", "StockCode": "string", "Description": "string", "Country": "category"},
).rename(columns={
    "InvoiceNo": "order_id", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "order_time", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
})
large_orders["sales"] = (large_orders["quantity"] * large_orders["unit_price"]).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C") | (large_orders["quantity"] < 0),
    "取消/退货", "完成"
)
print(f"UCI Online Retail 公开数据：{len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print("内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
large_orders.head()


In [ ]:
quality = pd.DataFrame({
    "缺失数": large_orders.isna().sum(),
    "缺失率": large_orders.isna().mean(),
    "唯一值": large_orders.nunique(dropna=False),
}).sort_values("缺失率", ascending=False)
print("完全重复行：", large_orders.duplicated().sum())
print("取消/退货行：", (large_orders["status"] == "取消/退货").sum())
display(quality.head(8))
clean_orders = large_orders.drop_duplicates().query("quantity > 0 and unit_price > 0").dropna(subset=["description"])
print(f"清洗后保留：{len(clean_orders):,} / {len(large_orders):,} 行")


## 常见误区

- 看到缺失值就全部填0
- 删除重复时未说明唯一键
- 把真实的大额订单误判为错误数据


## 综合练习

1. 创建含缺失和重复的客户表
2. 按客户ID去重
3. 使用中位数填充年龄并输出质量报告

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“创建含缺失和重复的客户表”。
2. **独立完成**：不复制示例代码，完成“按客户ID去重”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“使用中位数填充年龄并输出质量报告”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import numpy as np
import pandas as pd


customers = pd.DataFrame({
    "customer_id": ["U1", "U2", "U2", "U3"],
    "age": [28, np.nan, np.nan, 42],
    "city": ["上海", "广州", "广州", None],
})

# TODO: 按客户ID去重
clean = customers.drop_duplicates("customer_id").copy()

# TODO: 使用中位数填充年龄
clean["age"] =

# TODO: 填充城市缺失值为"未知"
clean["city"] =

print(clean)
print(clean.isna().sum())


In [ ]:
import numpy as np
import pandas as pd


customers = pd.DataFrame({
    "customer_id": ["U1", "U2", "U2", "U3"],
    "age": [28, np.nan, np.nan, 42],
    "city": ["上海", "广州", "广州", None],
})
clean = customers.drop_duplicates("customer_id").copy()
clean["age"] = clean["age"].fillna(clean["age"].median())
clean["city"] = clean["city"].fillna("未知")
print(clean)
print(clean.isna().sum())

# 自检
assert len(clean) == 3, "检查去重后行数：应该有3个唯一客户"
assert clean.isna().sum().sum() == 0, "检查缺失值：应该全部填充完成"


## 本章小结

建立数据质量检查流程，处理缺失、重复、异常和无效记录。

**迁移思考**：

1. 如果一个订单表中订单ID不重复，但同一用户有多个订单，去重时应该用什么键？
2. 为什么异常值应该先标记而不是直接删除？什么情况下可以删除异常值？


### 你已经掌握

- 生成质量概览
- 处理缺失值
- 识别并删除重复
- 使用规则标记异常


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 质量概览 | 同时查看形状、缺失、重复和类型。 | `pd.DataFrame()`、`orders.isna()`、`orders.duplicated()`、`.sum()` |
| 缺失与重复处理 | 订单ID重复时需要明确保留规则。 | `orders.drop_duplicates()`、`.copy()`、`.fillna()`、`.median()` |
| IQR异常标记 | 标记异常并保留原值，便于后续调查。 | `.quantile()`、`clean["amount"]`、`clean["is_outlier"]`、`clean[clean["is_outlier"]` |


### 需要注意

- 看到缺失值就全部填0
- 删除重复时未说明唯一键
- 把真实的大额订单误判为错误数据


### 完成检查

- [ ] 能够生成质量概览
- [ ] 能够处理缺失值
- [ ] 能够识别并删除重复
- [ ] 能够使用规则标记异常


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
